In [1]:
import hashlib; print(int(hashlib.md5("Минаков Данид".encode()).hexdigest()[:2], 16) % 8 + 1)

5


Вариант 5
Самые рейтинговые жанры сериалов

Задача: определить жанры сериалов с лучшим рейтингом.

Получить

| genre | avg_rating | series_count |

In [2]:
!pip install pyspark -q
!wget https://datasets.imdbws.com/title.basics.tsv.gz -q
!wget https://datasets.imdbws.com/title.crew.tsv.gz -q
!wget https://datasets.imdbws.com/name.basics.tsv.gz -q
!wget https://datasets.imdbws.com/title.ratings.tsv.gz -q

In [3]:
import pyspark.sql as pyspark

spark = (pyspark.SparkSession.builder
    .master('local')
    .appName("My Spark Application")
    .config('spark.executor.memory', '4g')
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

## **смотрим данные**

In [4]:
basics_df = spark.read.csv("title.basics.tsv.gz", sep='\t', header=True, nullValue='\\N')
basics_df.show(5)

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0000001|    short|          Carmencita|          Carmencita|      0|     1894|   NULL|             1|   Documentary,Short|
|tt0000002|    short|Le clown et ses c...|Le clown et ses c...|      0|     1892|   NULL|             5|     Animation,Short|
|tt0000003|    short|        Poor Pierrot|      Pauvre Pierrot|      0|     1892|   NULL|             5|Animation,Comedy,...|
|tt0000004|    short|         Un bon bock|         Un bon bock|      0|     1892|   NULL|            12|     Animation,Short|
|tt0000005|    short|    Blacksmith Scene|    Blacksmith Scene|      0|     1893|   NULL|             1|              

In [5]:
crew_df = spark.read.csv("title.crew.tsv.gz", sep='\t', header=True, nullValue='\\N')
crew_df.show(5)

+---------+---------+---------+
|   tconst|directors|  writers|
+---------+---------+---------+
|tt0000001|nm0005690|     NULL|
|tt0000002|nm0721526|     NULL|
|tt0000003|nm0721526|nm0721526|
|tt0000004|nm0721526|     NULL|
|tt0000005|nm0005690|     NULL|
+---------+---------+---------+
only showing top 5 rows


In [6]:
names_df = spark.read.csv("name.basics.tsv.gz", sep='\t', header=True, nullValue='\\N')
names_df.show(5)

+---------+---------------+---------+---------+--------------------+--------------------+
|   nconst|    primaryName|birthYear|deathYear|   primaryProfession|      knownForTitles|
+---------+---------------+---------+---------+--------------------+--------------------+
|nm0000001|   Fred Astaire|     1899|     1987|actor,miscellaneo...|tt0072308,tt00504...|
|nm0000002|  Lauren Bacall|     1924|     2014|actress,miscellan...|tt0037382,tt00752...|
|nm0000003|Brigitte Bardot|     1934|     2025|actress,music_dep...|tt0057345,tt00491...|
|nm0000004|   John Belushi|     1949|     1982|actor,writer,musi...|tt0072562,tt00779...|
|nm0000005| Ingmar Bergman|     1918|     2007|writer,director,a...|tt0050986,tt00694...|
+---------+---------------+---------+---------+--------------------+--------------------+
only showing top 5 rows


In [7]:
ratings_df = spark.read.csv("title.ratings.tsv.gz", sep='\t', header=True, nullValue='\\N')
ratings_df.show(5)

+---------+-------------+--------+
|   tconst|averageRating|numVotes|
+---------+-------------+--------+
|tt0000001|          5.7|    2201|
|tt0000002|          5.5|     312|
|tt0000003|          6.4|    2312|
|tt0000004|          5.1|     197|
|tt0000005|          6.2|    3038|
+---------+-------------+--------+
only showing top 5 rows


## **Задача**

In [8]:
basics_df.select('titleType').distinct().show()

+------------+
|   titleType|
+------------+
|    tvSeries|
|tvMiniSeries|
|     tvMovie|
|     tvPilot|
|   tvEpisode|
|       movie|
|   tvSpecial|
|       video|
|   videoGame|
|     tvShort|
|       short|
+------------+



In [9]:
series_df = basics_df[basics_df["titleType"] == "tvSeries"].select("tconst", "genres")

series_df = series_df.join(
    ratings_df.select("tconst", "averageRating"),
    on="tconst",
    how="left"
).dropna()

series_df.show(10)

+---------+-------------------+-------------+
|   tconst|             genres|averageRating|
+---------+-------------------+-------------+
|tt0029778|            Western|          6.3|
|tt0035803|   Documentary,News|          8.2|
|tt0038276|          Talk-Show|          6.1|
|tt0039120|   Family,Game-Show|          3.1|
|tt0039122|       Comedy,Music|          3.2|
|tt0039123|              Drama|          7.9|
|tt0039125|Crime,Drama,Mystery|          5.4|
|tt0040021|              Drama|          6.9|
|tt0040028|Comedy,Family,Music|          6.0|
|tt0040030|   Family,Talk-Show|          3.6|
+---------+-------------------+-------------+
only showing top 10 rows


In [10]:
from pyspark.sql.functions import split, explode, avg, count, col

genres_exploded = series_df.withColumn(
    "genre", explode(split(col("genres"), ","))
).drop("genres")

genres_exploded.show(10)

+---------+-------------+-----------+
|   tconst|averageRating|      genre|
+---------+-------------+-----------+
|tt0029778|          6.3|    Western|
|tt0035803|          8.2|Documentary|
|tt0035803|          8.2|       News|
|tt0038276|          6.1|  Talk-Show|
|tt0039120|          3.1|     Family|
|tt0039120|          3.1|  Game-Show|
|tt0039122|          3.2|     Comedy|
|tt0039122|          3.2|      Music|
|tt0039123|          7.9|      Drama|
|tt0039125|          5.4|      Crime|
+---------+-------------+-----------+
only showing top 10 rows


In [11]:
genres_exploded.groupBy("genre").agg(
    avg("averageRating").alias("avg_rating"),
    count("*").alias("count")
).orderBy(col("avg_rating").desc()).show()


+-----------+------------------+-----+
|      genre|        avg_rating|count|
+-----------+------------------+-----+
|Documentary| 7.295799357548799|12141|
|    Western| 7.272260273972603|  292|
|    History| 7.249870214146663| 3082|
|  Biography| 7.189992057188254| 1259|
|        War| 7.130165912518854|  663|
|  Adventure| 7.029781237417769| 7451|
|      Short| 7.029159519725553| 1166|
|      Sport|6.9645764119601425| 2408|
|    Mystery|  6.95875706214689| 3717|
|     Action| 6.950047150747637| 7423|
|      Crime| 6.942044854881252| 7580|
|    Fantasy| 6.932745878339951| 3518|
|  Animation| 6.918782704489148|10893|
|     Family| 6.915608528988972| 8348|
|      Drama| 6.882677456181622|29782|
|   Thriller| 6.874316163410295| 2815|
|     Horror| 6.843402450518381| 2122|
|    Romance| 6.830261230468745| 8192|
|     Sci-Fi| 6.829893238434163| 1967|
|     Comedy| 6.828600112944738|31874|
+-----------+------------------+-----+
only showing top 20 rows


# **Доп. задание**

### **Придумываем фичи**

In [12]:
from pyspark.sql.functions import size, length, when, coalesce
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Убрали названия но добавили фичу "сколько слов в названии"
# хочется использовать фичу endYear для этого для всяких фильмов сделам конец равным началу
# Также добавим принак "долгий ли фильм"
df = basics_df.withColumn("words_in_title", size(split(col("primaryTitle"), " "))) \
              .drop("primaryTitle", "originalTitle") \
              .withColumn("endYear", coalesce(col("endYear"), col("startYear"))) \
              .withColumn("is_long", (col("runtimeMinutes") > 100).cast("int"))



# Сделанм бинарные колонки для жанров
all_genres_list = df.withColumn("genre_array", split(col("genres"), ",")).select(explode("genre_array").alias("genre")) \
                    .filter(col("genre").isNotNull()).distinct().collect()

for row in all_genres_list:
    genre = row["genre"]
    col_name = f"genre_{genre.replace(' ', '_').replace('-', '_')}"
    df = df.withColumn(col_name, when(col("genres").contains(genre), 1).otherwise(0))

#Закодируем Title
title_indexer = StringIndexer(inputCol="titleType", outputCol="titleType_index", handleInvalid="keep")
title_encoder = OneHotEncoder(inputCol="titleType_index", outputCol="titleType_ohe")
df = title_indexer.fit(df).transform(df)
df = title_encoder.fit(df).transform(df)

df.show(10)

+---------+---------+-------+---------+-------+--------------+--------------------+--------------+-------+-----------+-------------+--------------+---------------+-----------+---------+-----------------+----------------+------------+-------------+---------------+-----------+-------------+-------------+-------------+---------------+-----------+---------------+-----------+------------+-------------+---------------+------------+-----------+------------+---------------+------------+----------+---------------+--------------+
|   tconst|titleType|isAdult|startYear|endYear|runtimeMinutes|              genres|words_in_title|is_long|genre_Crime|genre_Romance|genre_Thriller|genre_Adventure|genre_Drama|genre_War|genre_Documentary|genre_Reality_TV|genre_Family|genre_Fantasy|genre_Game_Show|genre_Adult|genre_History|genre_Mystery|genre_Musical|genre_Animation|genre_Music|genre_Film_Noir|genre_Short|genre_Horror|genre_Western|genre_Biography|genre_Comedy|genre_Sport|genre_Action|genre_Talk_Show|genr

In [13]:
# Посчитаем возраст режиссера в момент выхода
# Берём только первого режиссёра и считаем его возраст
df_with_age = basics_df.join(
    crew_df.withColumn("director", split(col("directors"), ",")[0]), "tconst", "left"
).join(
    names_df, col("director") == names_df["nconst"], "left"
).select(
    basics_df["tconst"], (col("startYear").cast("int") - col("birthYear").cast("int")).alias("creator_age")
)

df = df.join(df_with_age, "tconst", "left")

df.select("tconst", "creator_age").show(10)

+---------+-----------+
|   tconst|creator_age|
+---------+-----------+
|tt0000002|         48|
|tt0000004|         48|
|tt0000008|         34|
|tt0000010|         31|
|tt0000001|         34|
|tt0000003|         48|
|tt0000005|         33|
|tt0000006|         34|
|tt0000007|         34|
|tt0000009|         35|
+---------+-----------+
only showing top 10 rows


In [ ]:
df = df.withColumn("startYear", col("startYear").cast("int"))
df = df.withColumn("endYear", col("endYear").cast("int"))
df = df.withColumn("runtimeMinutes", col("runtimeMinutes").cast("int"))
df = df.dropna()
df.printSchema()

### **Моделька**

In [26]:
from pyspark.sql.functions import col, avg, coalesce, lit
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

df_clean = df.filter(col("startYear").rlike("^[0-9]+$"))
df_clean = df_clean.filter(col("runtimeMinutes").rlike("^[0-9]+$"))
df_clean = df_clean.filter(col("creator_age").rlike("^[0-9]+$|^$"))

df_clean = df_clean.withColumn("startYear", col("startYear").cast("int"))
df_clean = df_clean.withColumn("runtimeMinutes", col("runtimeMinutes").cast("int"))
df_clean = df_clean.withColumn("creator_age", col("creator_age").cast("int"))

regression_df = df_clean.select(
    col("tconst"),
    col("words_in_title"),
    col("startYear"),
    col("runtimeMinutes"),
    col("creator_age"),
    col("is_long"),
    *[col(c) for c in df_clean.columns if c.startswith("genre_")]
)

ratings_clean = ratings_df.select(col("tconst"), col("averageRating").cast("double").alias("rating"))
regression_df = regression_df.join(ratings_clean, on="tconst", how="inner")
regression_df = regression_df.dropna()
print(f"Строк после очистки: {regression_df.count()}")

feature_cols = ["words_in_title", "startYear", "runtimeMinutes", "creator_age", "is_long"] + \
               [c for c in regression_df.columns if c.startswith("genre_")]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
rf = RandomForestRegressor(featuresCol="features", labelCol="rating", numTrees=100)
pipeline = Pipeline(stages=[assembler, rf])

train, test = regression_df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train)

predictions = model.transform(test)

evaluator = RegressionEvaluator(labelCol="rating", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"RMSE: {rmse:.4f}")

RMSE: 1.0174
